In [ ]:
import numpy as np
from scipy.fft import fft, fftfreq
from scipy.integrate import trapezoid
from tqdm.notebook import tqdm

import sys
if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from multitone import compute_crest_factor

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def compute_M(m, t, delta_w):
    M = np.zeros_like(t, dtype=complex)
    for n, mn in enumerate(m):
        M += mn * np.exp(1j * n * delta_w * t)
    return M

def compute_error(M, S):
    E = np.zeros_like(M)
    Ma = np.abs(M)
    idx = Ma >= S
    E[idx] = (Ma[idx] - S) * np.exp(1j * np.angle(M[idx]))
    return E

def plot_M(M, t, w0, N_tones):
    tmp = M * np.exp(1j * w0 * t)
    Mr, Mi = tmp.real, tmp.imag
    Ma = np.abs(M)
    fig,ax = plt.subplots(2, 1, figsize=(6,4), sharex=True)
    ax[0].plot(t, Ma)
    ax[0].hlines(np.sqrt(N_tones), 0, T, color='tab:red', ls='--', lw=2)
    ax[1].plot(t, Mr)
    ax[0].set_ylabel('M')
    ax[1].set_xlabel('Time [s]')
    ax[1].set_ylabel('Re{M * exp(j * ω0 * t)}')
    sns.despine()
    fig.tight_layout()

def optimize_m(t, delta_w, N_tones, N_iter, phi0=None):
    from scipy.fft import fft
    if phi0 is None:
        phi0 = np.random.uniform(0, 2 * np.pi, N_tones)
    S0 = 1.1 * np.sqrt(N_tones)
    CF = np.zeros(N_iter)
    S = np.zeros_like(CF)
    m = np.zeros((N_iter, N_tones), dtype=complex)
    S[0] = S0
    m[0] = np.exp(1j * phi0)
    dt = t[1] - t[0]
    N_samples = t.size
    for i in range(N_iter - 1):
        M = compute_M(m[i], t, delta_w)
        CF[i] = compute_crest_factor(M, dt)
        e = compute_error(M, S[i])
        ef = fft(e)[:N_tones] / e.size
        m[i + 1] = np.exp(1j * np.angle(m[i] - ef))
        S[i + 1] = min(CF[i] * np.sqrt(N_tones) * 0.95, S[i] * 1.001)
    i = N_iter - 1
    M = compute_M(m[i], t, delta_w)
    CF[i] = compute_crest_factor(M, dt)
    return m, CF, S

In [ ]:
F0 = 5e-3
T = 1 / F0
N_tones = 32
Δω = 2 * np.pi * F0
ω0 = 1 * Δω
dt = T / 5000
t = np.r_[0 : T + dt / 2 : dt]
print(f'F0 = {F0} Hz')
print(f'# of tones: {N_tones}')
print(f'T = {T} s')
print(f'dt = {dt} s')

In [ ]:
N_iter = 5000
CF_min = None
for i in tqdm(range(20)):
    m, CF, S = optimize_m(t, Δω, N_tones, N_iter)
    if CF_min is None or CF.min() < CF_min:
        CF_min = CF.min()
        m_opt = m
        CF_opt = CF
        S_opt = S

In [ ]:
fig,ax = plt.subplots(2, 1, figsize=(6,4), sharex=True)
ax[0].plot(CF_opt)
ax[1].plot(S_opt)
ax[1].set_xlabel('Iteration')
ax[0].set_ylabel('CF')
ax[1].set_ylabel('S')
sns.despine()
fig.tight_layout()

In [ ]:
idx, = np.where(CF_opt == CF_min)
M = compute_M(m_opt[idx].squeeze(), t, Δω)
Mr = (M * np.exp(1j * ω0 * t)).real
plot_M(M, t, 1 * ω0, N_tones)

In [ ]:
N_samples = Mr.size
Mrf = fft(Mr)
f = fftfreq(Mr.size, dt)[:N_samples//2]
Mrfps = 2 / N_samples * np.abs(Mrf)[:N_samples//2]

In [ ]:
fig,ax = plt.subplots(1, 1, figsize=(6,3))
ax.plot(f * 1e3, Mrfps, '.')
ax.set_xlim([0, (N_tones + 2) * F0 * 1e3])
ax.set_xlabel('Frequency [mHz]')
ax.set_ylabel('|FFT(m)|')
sns.despine()
fig.tight_layout()